# BBCad — TRELLIS Image-to-3D Service
Run this notebook on **Google Colab** (T4 GPU) or **Kaggle** (T4 GPU).

Steps:
1. Run all cells top to bottom
2. Copy the **ngrok URL** printed at the end
3. Paste it into your `.env` as `TRELLIS_SERVICE_URL=https://xxxx.ngrok-free.app`
4. Restart your Express server

> Keep this tab open — closing it kills the GPU session.

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'No GPU found — enable GPU runtime first!')

In [ ]:
# ── Cell 2: Install system deps ───────────────────────────────────────────
!apt-get install -y -qq libgl1 libglib2.0-0 libgomp1

In [ ]:
# ── Cell 3: Clone TRELLIS ─────────────────────────────────────────────────
import os
if not os.path.exists('/content/TRELLIS'):
    !git clone --depth=1 https://github.com/microsoft/TRELLIS.git /content/TRELLIS
else:
    print('TRELLIS already cloned')

In [ ]:
# ── Cell 4: Install TRELLIS deps ──────────────────────────────────────────
# This takes ~3-4 minutes
%cd /content/TRELLIS
!pip install -q -e . 2>&1 | tail -5
!pip install -q spconv-cu120 2>&1 | tail -3
!pip install -q xformers 2>&1 | tail -3
!pip install -q fastapi uvicorn pyngrok pillow 2>&1 | tail -3
print('Dependencies installed!')

In [ ]:
# ── Cell 5: Write the FastAPI service ─────────────────────────────────────
service_code = '''
import os, sys, uuid, base64, traceback, threading
from io import BytesIO
from pathlib import Path
from typing import Dict, Any

sys.path.insert(0, "/content/TRELLIS")

from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel

OUTPUT_DIR = Path("/content/trellis_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

tasks: Dict[str, Dict[str, Any]] = {}
tasks_lock = threading.Lock()
app = FastAPI()

_pipeline = None
_pipeline_lock = threading.Lock()

def get_pipeline():
    global _pipeline
    if _pipeline: return _pipeline
    with _pipeline_lock:
        if _pipeline: return _pipeline
        print("[TRELLIS] Loading model... (first time ~2 min)", flush=True)
        from trellis.pipelines import TrellisImageTo3DPipeline
        _pipeline = TrellisImageTo3DPipeline.from_pretrained("JeffreyXiang/TRELLIS-image-large")
        _pipeline.cuda()
        print("[TRELLIS] Model ready!", flush=True)
        return _pipeline

def _run(task_id, image_bytes, sparse_steps, slat_steps, seed):
    try:
        import torch
        from PIL import Image
        with tasks_lock:
            tasks[task_id]["status"] = "IN_PROGRESS"
            tasks[task_id]["progress"] = 10

        img = Image.open(BytesIO(image_bytes)).convert("RGBA")
        pipe = get_pipeline()

        with tasks_lock: tasks[task_id]["progress"] = 20

        with torch.inference_mode():
            outputs = pipe.run(
                img, seed=seed,
                sparse_structure_sampler_params={"steps": sparse_steps, "cfg_strength": 7.5},
                slat_sampler_params={"steps": slat_steps, "cfg_strength": 3.0},
            )

        with tasks_lock: tasks[task_id]["progress"] = 85

        glb_path = OUTPUT_DIR / f"{task_id}.glb"
        outputs[0].export(str(glb_path))

        with tasks_lock:
            tasks[task_id].update({"status": "SUCCEEDED", "progress": 100, "glb_path": str(glb_path)})
        print(f"[TRELLIS] Done: {glb_path}", flush=True)

    except Exception as e:
        traceback.print_exc()
        with tasks_lock:
            tasks[task_id].update({"status": "FAILED", "error": str(e)})

class GenReq(BaseModel):
    image_data: str
    sparse_steps: int = 25
    slat_steps: int = 25
    seed: int = 42

@app.get("/health")
def health(): return {"ok": True}

@app.post("/generate")
def generate(req: GenReq):
    try:
        image_bytes = base64.b64decode(req.image_data)
    except Exception:
        raise HTTPException(400, "Invalid base64")
    task_id = str(uuid.uuid4())
    with tasks_lock:
        tasks[task_id] = {"status": "PENDING", "progress": 0, "glb_path": None, "error": None}
    threading.Thread(target=_run, args=(task_id, image_bytes, req.sparse_steps, req.slat_steps, req.seed), daemon=True).start()
    return {"task_id": task_id}

@app.get("/status/{task_id}")
def status(task_id: str):
    with tasks_lock: task = tasks.get(task_id)
    if not task: raise HTTPException(404, "Not found")
    resp = {"id": task_id, "status": task["status"], "progress": task["progress"]}
    if task["status"] == "SUCCEEDED":
        resp["model_urls"] = {"glb": f"/result/{task_id}"}
    if task["error"]:
        resp["task_error"] = {"message": task["error"]}
    return resp

@app.get("/result/{task_id}")
def result(task_id: str):
    with tasks_lock: task = tasks.get(task_id)
    if not task or task["status"] != "SUCCEEDED": raise HTTPException(400, "Not ready")
    return FileResponse(task["glb_path"], media_type="model/gltf-binary", filename=f"{task_id}.glb")
'''

with open('/content/trellis_api.py', 'w') as f:
    f.write(service_code)
print('Service file written!')

In [ ]:
# ── Cell 6: Get ngrok auth token ──────────────────────────────────────────
# Sign up free at https://ngrok.com → Dashboard → Your Authtoken
# Paste your token below:
NGROK_TOKEN = ""  # <-- paste your ngrok token here

if not NGROK_TOKEN:
    print("ERROR: Paste your ngrok token above!")
    print("Get it free at: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    print('ngrok token set!')

In [ ]:
# ── Cell 7: Start server + ngrok tunnel ───────────────────────────────────
import subprocess, time, requests
from pyngrok import ngrok

# Start FastAPI in background
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'trellis_api:app', '--host', '0.0.0.0', '--port', '8765'],
    cwd='/content',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

# Wait for it to start
time.sleep(3)

# Open ngrok tunnel
tunnel = ngrok.connect(8765)
public_url = tunnel.public_url

print('=' * 60)
print('SERVICE IS LIVE!')
print('=' * 60)
print()
print(f'ngrok URL: {public_url}')
print()
print('Add this to your .env file:')
print(f'TRELLIS_SERVICE_URL={public_url}')
print()
print('Then restart your Express server.')
print('Keep this notebook open!')

# Health check
try:
    r = requests.get(f'{public_url}/health', timeout=5)
    print(f'\nHealth check: {r.json()}')
except Exception as e:
    print(f'Health check failed: {e} — wait a moment and retry')

In [ ]:
# ── Cell 8 (optional): Keep-alive — run this to prevent Colab timeout ────
# Colab disconnects after ~90 min of inactivity.
# This cell pings the service every 30s to keep it alive.
import time, requests, threading

def keepalive():
    while True:
        try:
            requests.get(f'{public_url}/health', timeout=5)
        except: pass
        time.sleep(30)

t = threading.Thread(target=keepalive, daemon=True)
t.start()
print('Keep-alive running. This cell will keep looping — that is normal.')
# Interrupt kernel to stop